# Lab: Build a Transformer Encoder Block

**Bluevision-ai Academy**
*The Transformer Architecture: Multi-Head Attention, Positional Encoding, and the Encoder Block*

Session 04's lab built scaled dot-product attention from scratch and then proved, with a bit-exact measurement, that attention alone cannot tell "the cat sat down" from "sat the down cat" -- permuting the input permutes the output by exactly the same amount, no more, no less. Today's lecture assembled the rest of the machine around that primitive: run several attention heads in parallel, wrap the result in a feed-forward layer with residual connections and layer normalisation, and inject a fixed sine/cosine signal so position is no longer invisible. This lab builds all of it -- multi-head attention, the full encoder block, and real positional encodings -- and then asks the one question last session's lab left open: does any of this actually fix the order-blindness, and by how much?

## Setup

Real per-word embeddings come from the same pretrained multilingual encoder session 04's lab used, so today's numbers are directly comparable to last session's. Nothing in this notebook is trained -- every block below runs forward passes only, on its own randomly initialised weights.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade torch transformers plotly

import numpy as np
import torch
import torch.nn as nn
import plotly.graph_objects as go

torch.manual_seed(0)
np.random.seed(0)

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#5c6b78"

In [2]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)
encoder.eval()

D_MODEL = 768  # this encoder's hidden size -- also BERT-base's, session 06's own d_model

def word_embeddings(words):
    """One real contextual vector per word, mean-pooled over its subword pieces
    (same helper session 04's lab used)."""
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt")
    with torch.no_grad():
        hidden = encoder(**enc).last_hidden_state[0]
    word_ids = enc.word_ids()
    return torch.stack([hidden[[i for i, w in enumerate(word_ids) if w == wi]].mean(dim=0)
                         for wi in range(len(words))])

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
sentence_a = ["the", "cat", "sat", "down"]
sentence_b = ["a", "dog", "ran", "away"]

emb_a = word_embeddings(sentence_a)
emb_b = word_embeddings(sentence_b)
batch = torch.stack([emb_a, emb_b])   # (B=2, T=4, D_MODEL)

print(f"sentence_a: {sentence_a}")
print(f"sentence_b: {sentence_b}")
print(f"batch shape: {tuple(batch.shape)}")

sentence_a: ['the', 'cat', 'sat', 'down']
sentence_b: ['a', 'dog', 'ran', 'away']
batch shape: (2, 4, 768)


## Part 1 -- Batched Scaled Dot-Product Attention

**Problem.** Session 04's `attention(Q, K, V)` took a single query/key/value matrix -- one sentence, no batch dimension. Everything from here on needs more than that: Part 2 runs H attention computations at once, one per head, and a real Transformer processes many sentences in a single forward pass, which is the entire point of throwing out the RNN (today's opening slides). Before building anything new, the primitive itself has to carry an extra dimension without changing what it computes.

**Ingredients.** The real two-sentence batch built in Setup (`batch`, shape `(2, 4, 768)`) -- used here only to prove shapes flow correctly, the same "nothing external yet" spirit as session 04's Part 1.

**Idea -- generalise the formula, don't rederive it.** Slide 10 already wrote the mechanism as one line, `MultiHead(Q,K,V) = Concat(head_1,...,head_H) W^O`, and every head inside it is the same `softmax(QK^T/sqrt(d_k))V` from session 04. The design choice here is only how to add the batch dimension: loop over it in Python (simple, one call per example) or let PyTorch's `@` operator broadcast over any number of leading dimensions in a single call. Real implementations use the second option, because that is what actually lets a GPU process a whole batch -- and later, a whole set of heads -- at once; that parallelism is the mechanical reason attention replaced the RNN. Implement it that way, swapping `.T` for `.transpose(-2, -1)` so the same function works whether the leading shape is `(T, d)`, `(B, T, d)`, or, in Part 2, `(B, H, T, d)`.

In [4]:
def attention(Q, K, V):
    """Q: (..., n_q, d_k)  K, V: (..., n_k, d_k)  ->  output: (..., n_q, d_v), weights: (..., n_q, n_k)
    Any number of leading batch dimensions -- unbatched, batched, or per-head."""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / d_k ** 0.5
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

**Idea -- trust, but verify, on two examples at once.** Session 04 checked correctness with one toy example: three orthogonal keys, a query built to align with exactly one of them. Do the same thing again here, but pack **two independent toy problems into a single batched call**, each with its own query aligned to a *different* key. If the batched implementation is correct, example 0's weights must land on key 1 and example 1's weights must land on key 2, with zero leakage between them -- any cross-contamination would mean the batch dimension is silently mixing examples together.

In [5]:
K_toy = torch.eye(3, 4).unsqueeze(0).repeat(2, 1, 1)                     # (2, 3, 4): same 3 orthogonal keys in both examples
V_toy = torch.tensor([[10., 0, 0, 0],
                       [0, 20., 0, 0],
                       [0, 0, 30., 0]]).unsqueeze(0).repeat(2, 1, 1)      # (2, 3, 4): a distinct, checkable value per key
Q_toy = torch.tensor([[[0., 20., 0., 0.]],     # example 0's query aligns with key 1
                       [[0., 0., 20., 0.]]])    # example 1's query aligns with key 2

out_toy, w_toy = attention(Q_toy, K_toy, V_toy)
print("weights (2 examples x 1 query x 3 keys):")
print(w_toy.numpy().round(4))
print("outputs:")
print(out_toy.numpy().round(2))

weights (2 examples x 1 query x 3 keys):
[[[0.     0.9999 0.    ]]

 [[0.     0.     0.9999]]]
outputs:
[[[ 0. 20.  0.  0.]]

 [[ 0.  0. 30.  0.]]]


**Idea -- now the real batch, self-attention, checked against a Python loop.** With the toy case passing, run the same function as genuine self-attention (`Q=K=V=batch`) over the two real sentences, and compare the one-call batched result against a reference that loops over the batch in plain Python, one sentence at a time. If both give the same answer, the parallel version is not just faster -- it is provably the same computation.

In [6]:
out_real, w_real = attention(batch, batch, batch)
print(f"output shape: {tuple(out_real.shape)}  weights shape: {tuple(w_real.shape)}")
print("each weight row sums to 1 (a real probability distribution):")
print(w_real.sum(dim=-1))

looped = torch.stack([attention(batch[b], batch[b], batch[b])[0] for b in range(batch.shape[0])])
print(f"\nmax |batched - looped| = {(looped - out_real).abs().max().item():.2e}")

output shape: (2, 4, 768)  weights shape: (2, 4, 4)
each weight row sums to 1 (a real probability distribution):
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]])

max |batched - looped| = 0.00e+00


**Payoff.** Every check passes cleanly: the two toy examples land on their own intended key with no leakage (0.9999 on the right key, ~0 elsewhere, in each batch entry independently), the real batch produces the expected `(2, 4, 768)` output with weight rows summing to exactly 1, and the batched call matches the looped reference to floating-point precision. That last number matters more than it looks: it is a real proof that moving the batch into the matmul, instead of looping over it in Python, changes *nothing about the answer*, only how the computation gets there -- looping and broadcasting compute the identical function, and a production Transformer picks broadcasting purely for throughput. The limitation this primitive still carries: every query compares against every key using the entire 768-dimensional vector at once, so any two kinds of relationship a word might have -- its subject, its topic, its position -- get folded into a single weighting. Slide 10 asked "why isn't one enough?"; Part 2 answers it by giving the model several of these comparisons to run side by side, each in its own smaller subspace.

## Part 2 -- Multi-Head Attention From Scratch

**Problem.** A single attention computation over all 768 dimensions forces the model to average every kind of relationship -- syntax, reference, position -- into one weighting per token pair. Slide 10's fix: split the model dimension into `H` smaller subspaces and run an independent attention computation in each, so the model has room to learn several relationships side by side, exactly as slide 11 showed with one trained head tracking "sat" locally and another tracking subject-verb agreement. Build the mechanism -- no `nn.MultiheadAttention` -- and then check, on this notebook's own *untrained* weights, whether that specialisation is something the architecture hands you for free.

**Ingredients.** `H = 12`, `d_k = d_v = 768 / 12 = 64` -- real BERT-base's own configuration, not an arbitrary choice, since `D_MODEL = 768` was already fixed by the pretrained encoder in Setup. The real batch from Setup.

**Idea -- one linear projection per Q/K/V, then reshape into heads.** Slide 10's diagram draws H separate small projections, but a single `nn.Linear(768, 768)` followed by a reshape does the identical computation in one matrix multiply instead of H small ones -- the same batching trade-off Part 1 just made, applied one level up. Split the 768 output columns into `(H, d_k)` and move the head axis next to the batch axis, so `attention()` from Part 1, completely unchanged, now runs all H heads in parallel as just another leading dimension.

In [7]:
H = 12
D_K = D_MODEL // H
print(f"H = {H} heads, d_k = d_v = {D_K}")

def split_heads(x):
    B, T, D = x.shape
    return x.view(B, T, H, D_K).transpose(1, 2)                     # (B, H, T, d_k)

def merge_heads(x):
    B, Hh, T, dk = x.shape
    return x.transpose(1, 2).contiguous().view(B, T, Hh * dk)       # (B, T, D)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        Qp, Kp, Vp = self.Wq(x), self.Wk(x), self.Wv(x)
        out, weights = attention(split_heads(Qp), split_heads(Kp), split_heads(Vp))
        return self.Wo(merge_heads(out)), weights

torch.manual_seed(0)
mha = MultiHeadAttention(D_MODEL, H)
mha.eval()

H = 12 heads, d_k = d_v = 64


MultiHeadAttention(
  (Wq): Linear(in_features=768, out_features=768, bias=True)
  (Wk): Linear(in_features=768, out_features=768, bias=True)
  (Wv): Linear(in_features=768, out_features=768, bias=True)
  (Wo): Linear(in_features=768, out_features=768, bias=True)
)

**Idea -- verify the reshape trick against a literal per-head loop.** Slide 10's own diagram runs H heads side by side, then concatenates them. Confirm the reshape-based version above computes exactly that: project Q, K, V once, then loop over the H head-slices in plain Python, run `attention()` on each slice separately, concatenate, and apply the same `W^O`. Both paths must agree, since they compute the same function two different ways.

In [8]:
with torch.no_grad():
    mha_out, mha_w = mha(batch)

    Qp, Kp, Vp = mha.Wq(batch), mha.Wk(batch), mha.Wv(batch)
    Qh, Kh, Vh = split_heads(Qp), split_heads(Kp), split_heads(Vp)
    head_outs = [attention(Qh[:, h], Kh[:, h], Vh[:, h])[0] for h in range(H)]
    looped_out = mha.Wo(torch.cat(head_outs, dim=-1))

print(f"mha output shape: {tuple(mha_out.shape)}  attention-weight shape: {tuple(mha_w.shape)}")
print(f"max |reshape - looped| = {(looped_out - mha_out).abs().max().item():.2e}")

mha output shape: (2, 4, 768)  attention-weight shape: (2, 12, 4, 4)
max |reshape - looped| = 0.00e+00


**Idea -- test slide 11's claim on weights that have never been trained.** Slide 11 showed a *trained* Transformer where the query "sat" split cleanly across two heads: one 70% on itself, another 75% on "cat". `mha` above has never seen a single training example -- its `W^Q`/`W^K`/`W^V` are only random initial values. Pull out every head's attention distribution for the query "sat" (token index 2 of `sentence_a`) and check whether anything like slide 11's clean split shows up already, purely from having 12 differently-initialised heads.

In [9]:
sat_weights = mha_w[0, :, 2, :]          # sentence_a is batch index 0, "sat" is token index 2
max_per_head = sat_weights.max(dim=-1).values

print("attention('sat') across all 12 heads, over [the, cat, sat, down]:")
print(sat_weights.detach().numpy().round(3))
print(f"\nmax weight per head: {max_per_head.detach().numpy().round(3)}")
print(f"mean of those maxima:  {max_per_head.mean().item():.4f}")
print(f"uniform baseline (1/4): {0.25:.4f}")

fig = go.Figure()
fig.add_trace(go.Bar(x=[f"head {h+1}" for h in range(H)], y=max_per_head.detach().numpy(),
                      marker_color=GOLD, name="max attention weight"))
fig.add_hline(y=0.25, line_dash="dash", line_color=DIM,
              annotation_text="uniform (1/4)", annotation_position="top left")
fig.update_layout(title="Untrained heads: how far from uniform is each head's 'sat' query?",
                   yaxis_title="max attention weight", yaxis_range=[0, 0.8],
                   template="plotly_white", height=380)
fig.show()

attention('sat') across all 12 heads, over [the, cat, sat, down]:
[[0.252 0.257 0.245 0.247]
 [0.252 0.256 0.249 0.242]
 [0.255 0.256 0.233 0.256]
 [0.245 0.26  0.245 0.25 ]
 [0.26  0.247 0.247 0.246]
 [0.254 0.248 0.251 0.247]
 [0.253 0.247 0.248 0.252]
 [0.26  0.228 0.247 0.265]
 [0.252 0.253 0.246 0.249]
 [0.244 0.248 0.255 0.252]
 [0.253 0.251 0.246 0.25 ]
 [0.259 0.25  0.243 0.248]]

max weight per head: [0.257 0.256 0.256 0.26  0.26  0.254 0.253 0.265 0.253 0.255 0.253 0.259]
mean of those maxima:  0.2567
uniform baseline (1/4): 0.2500


**Payoff.** Both correctness checks pass: `mha`'s reshape-based forward pass agrees with the literal per-head loop to floating-point precision, and the output shape is the expected `(2, 4, 768)`. The "sat" query tells the more interesting story. Every one of the 12 heads sits within about a percentage point of the uniform baseline (mean max weight 0.257 against 0.25), nothing close to slide 11's 70% and 75% peaks. The heads *are* structurally independent, each with its own randomly initialised `W^Q`/`W^K`/`W^V`, provably by construction; but independence is not the same thing as specialisation. A head only learns to track subject-verb agreement, or anything else, once gradient descent has a loss pushing it there -- the architecture supplies the *capacity* for 12 different relationships, training supplies which 12 those turn out to be. That is the same optimisation-versus-capacity distinction session 04's lab drew between a correct implementation and a good one, one level up: a multi-head layer can be built correctly and still be doing nothing useful yet.

## Part 3 -- The Full Encoder Block

**Problem.** Multi-head attention is one sub-layer. Slide 14 assembled the complete unit -- attention, then a feed-forward network, each wrapped in `LayerNorm(x + Sublayer(x))` -- and slide 15 showed why that specific unit matters: it gets stacked N times, 6 in the original paper, 12 in BERT-base, 96 in GPT-3, always the same shape in and out. Build that reusable block, then verify directly the one property that makes "just stack more of them" a meaningful lever at all: does the output shape survive being stacked?

**Ingredients.** The `MultiHeadAttention` class from Part 2. `D_FF = 4 * 768 = 3072`, the same "expand then contract" ratio slide 12 named. The real batch from Setup.

**Idea -- compose, don't re-derive.** Parts 1 and 2 hand-built the two mechanisms the lecture derived from first principles: attention and multi-head attention. `LayerNorm` and a two-layer MLP are not new mechanisms this session introduced -- they are standard building blocks (slide 13's `output = LayerNorm(x + Sublayer(x))` names the pattern, it does not derive it) -- so use PyTorch's own `nn.LayerNorm` and `nn.Linear` for those, and spend the engineering effort on getting the *composition* right: two sub-layers, each with its own residual bypass and its own norm, in the exact order slide 14 walked through.

In [10]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ff)
        self.W2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.W2(torch.relu(self.W1(x)))            # FFN(x) = max(0, xW1+b1)W2+b2, slide 12

class EncoderBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, h)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.mha(x)
        x = self.norm1(x + attn_out)                       # sub-layer 1: attention + residual + norm
        x = self.norm2(x + self.ffn(x))                    # sub-layer 2: feed-forward + residual + norm
        return x

D_FF = 4 * D_MODEL

**Idea -- prove stackability, not just build one block.** Instantiate two independently-initialised encoder blocks and run the real batch through both in sequence, `block1` then `block2`. If `EncoderBlock` really is the interchangeable unit slide 15 draws it as, the shape must be identical after the input, after block 1, and after block 2, with nothing in the code that ties it to being "the first" or "the last" block in a stack.

In [11]:
torch.manual_seed(1)
block1 = EncoderBlock(D_MODEL, H, D_FF)
block2 = EncoderBlock(D_MODEL, H, D_FF)
block1.eval(); block2.eval()

with torch.no_grad():
    h1 = block1(batch)
    h2 = block2(h1)

print(f"input:          {tuple(batch.shape)}")
print(f"after block 1:  {tuple(h1.shape)}")
print(f"after block 2:  {tuple(h2.shape)}")

input:          (2, 4, 768)
after block 1:  (2, 4, 768)
after block 2:  (2, 4, 768)


**Idea -- shape-preserving is not the same as doing nothing.** Passing the shape check would also be true of a block that just returned its input unchanged. Measure how much the representation actually moved: cosine similarity between each token's input embedding and its representation after two full blocks. A value near 1.0 would mean the block barely touched anything; a value near 0 would mean it destroyed the original signal.

In [12]:
cos = nn.functional.cosine_similarity(batch, h2, dim=-1)
print("cosine similarity, input vs. after 2 stacked blocks (per token):")
for words, row in zip([sentence_a, sentence_b], cos):
    for w, c in zip(words, row):
        print(f"  {w:6s} {c.item():.4f}")
print(f"\nmean: {cos.mean().item():.4f}")

cosine similarity, input vs. after 2 stacked blocks (per token):
  the    0.8612
  cat    0.8671
  sat    0.8675
  down   0.8699
  a      0.8860
  dog    0.8817
  ran    0.8816
  away   0.8871

mean: 0.8753


**Payoff.** The shape survives exactly: `(2, 4, 768)` in, `(2, 4, 768)` after one block, `(2, 4, 768)` after two -- and nothing in `EncoderBlock` refers to how many blocks came before it, which is the literal mechanism behind slide 15's "just stack N of them," whether N is 6 or 96. The cosine similarities, all clustering around 0.86 to 0.89, rule out the trivial case: two full blocks move every token's representation a real, substantial distance from where it started, not a rounding error. What the notebook cannot yet tell you is whether *that particular* movement is useful. With random initialisation, `EncoderBlock` is a real, shape-preserving, non-trivial transformation, and, same as Part 2's untrained heads, an unguided one. The architecture guarantees composability; only training turns composability into representations that are actually good for something, which is exactly where session 06 picks up.

## Challenge -- Break It Deliberately: Does the Encoder Block Actually Fix the Order-Blindness?

**Problem.** Session 04's lab proved something exact, not approximate: permuting the words fed into bare `attention()` permutes the output by exactly the same amount, max difference 0.00e+00. Today's lecture argued that adding positional encoding fixes this. Two real questions follow, and this notebook can answer both precisely instead of taking either on faith: does the blindness survive all the way through the *full* encoder block -- `LayerNorm` and the feed-forward network are also applied identically at every position, so they might not break the symmetry either -- and does the real sine/cosine fix from slide 9 actually move the needle, measured the same way session 04 measured its own crude patch?

**Ingredients.** `sentence_a` and its embedding `emb_a` from Setup, and a fresh `EncoderBlock`. Slide 9's formula, implemented directly.

**Idea -- write the formula slide 9 already derived, don't re-derive it.** `PE(pos, 2i) = sin(pos / 10000^(2i/d))`, odd dimensions cosine instead of sine. No new reasoning is needed here; the lecture's job was explaining *why* this specific shape (bounded, length-independent, relative offsets recoverable), and this cell's job is only to compute it.

In [13]:
def pos_encoding(T, D):
    """PE(pos, 2i) = sin(pos / 10000^(2i/D)); PE(pos, 2i+1) = cos(pos / 10000^(2i/D)) -- slide 9."""
    pos = torch.arange(T).unsqueeze(1).float()
    i = torch.arange(D).unsqueeze(0).float()
    angles = pos / (10000 ** (2 * (i // 2) / D))
    pe = torch.zeros(T, D)
    pe[:, 0::2] = torch.sin(angles[:, 0::2])
    pe[:, 1::2] = torch.cos(angles[:, 1::2])
    return pe

T = emb_a.shape[0]
PE = pos_encoding(T, D_MODEL)
print(f"PE shape: {tuple(PE.shape)}  range: [{PE.min().item():.3f}, {PE.max().item():.3f}]")

PE shape: (4, 768)  range: [-0.990, 1.000]


**Idea -- session 04's exact measurement, now on the full block, without PE first.** Repeat session 04's methodology precisely, so the two sessions' numbers are directly comparable: permute the four words, run both the original order and the permuted order through a fresh `EncoderBlock`, and compare `output(permuted)` against `permutation applied to output(original)`. If the block is still permutation-equivariant, those two must match exactly.

In [14]:
perm = torch.tensor([2, 0, 3, 1])
sentence_a_perm = [sentence_a[i] for i in perm.tolist()]
print(f"original:  {sentence_a}")
print(f"permuted:  {sentence_a_perm}")

torch.manual_seed(2)
test_block = EncoderBlock(D_MODEL, H, D_FF)
test_block.eval()

with torch.no_grad():
    out_orig_noPE = test_block(emb_a.unsqueeze(0))[0]
    out_perm_noPE = test_block(emb_a[perm].unsqueeze(0))[0]

diff_noPE = (out_perm_noPE - out_orig_noPE[perm]).abs().max().item()
print(f"\nmax |block(permuted) - permute(block(original))|, no PE: {diff_noPE:.2e}")

original:  ['the', 'cat', 'sat', 'down']
permuted:  ['sat', 'the', 'down', 'cat']

max |block(permuted) - permute(block(original))|, no PE: 9.54e-07


**Idea -- the identical test, now with positional encoding added first.** Add `PE` to the embeddings *before* the block sees them, and permute the words the same way as before -- crucially, permute first, then add `PE` for the new slot order, so each word's positional signal reflects where it actually sits after shuffling. If positional encoding does what slide 9 claims, this same measurement should no longer come out near zero.

In [15]:
with torch.no_grad():
    out_orig_PE = test_block((emb_a + PE).unsqueeze(0))[0]
    out_perm_PE = test_block((emb_a[perm] + PE).unsqueeze(0))[0]   # PE added AFTER permuting -> tied to the new slot

diff_PE = (out_perm_PE - out_orig_PE[perm]).abs().max().item()
print(f"max |block(permuted) - permute(block(original))|, with PE: {diff_PE:.2e}")
print(f"\nratio (with PE) / (no PE): {diff_PE / diff_noPE:,.0f}x")
print(f"ratio to session 04's own crude patch (2.54e-02): {diff_PE / 2.54e-2:.1f}x")

fig = go.Figure(go.Bar(x=["no positional<br>encoding", "with positional<br>encoding",
                          "session 04's crude<br>slot-index patch"],
                       y=[diff_noPE, diff_PE, 2.54e-2],
                       marker_color=[DIM, GOLD, BLUE]))
fig.update_layout(title="How much does word order change the output? (max abs. difference, log scale)",
                   yaxis_type="log", yaxis_title="max |difference|",
                   template="plotly_white", height=380)
fig.show()

max |block(permuted) - permute(block(original))|, with PE: 2.25e+00

ratio (with PE) / (no PE): 2,354,746x
ratio to session 04's own crude patch (2.54e-02): 88.4x


**Payoff.** Without positional encoding, the max difference is 9.54e-07, floating-point noise, not a real number -- the same 0.00e+00 conclusion session 04 reached for bare attention, now shown to survive `LayerNorm` and the feed-forward network too, since both apply the identical operation at every position and so neither breaks the symmetry attention already had. With positional encoding added, the max difference jumps to roughly 2.25, more than two million times larger, and about 88 times the effect of session 04's own deliberately crude fix (adding `0.5 * slot_index` to each embedding). That is a real, quantified answer to last session's open question: the sine/cosine signal does not just theoretically satisfy slide 9's three requirements, it produces a dramatically larger, more decisive break in permutation symmetry than the cheapest patch that could pass the same test. The honest limit of what this proves: a bigger number here only shows that word order now *changes* the output substantially -- it says nothing about whether that change is the *right* one. An untrained block, same as Parts 2 and 3, has no way to know that "cat sat" and "sat cat" should mean different things beyond producing different vectors for them; teaching it that difference is what training does, and that is exactly where session 06 picks up.

## Reflection

**Problem.** You now have a real, from-scratch encoder block: batched attention, multi-head attention, the residual-and-norm wrapping, and a measured, not argued, account of what positional encoding buys you. This last cell has no code to check; it asks you to put the reasoning in your own words.

In [16]:
# 1. Part 1 proved the batched attention() matches a looped-over-the-batch
#    reference exactly (max diff 0.00e+00). Part 2 proved the reshape-into-heads
#    MultiHeadAttention matches a looped-over-the-heads reference exactly too.
#    What real engineering property do these two checks establish, and why
#    does a from-scratch implementation need to prove it explicitly rather
#    than just assuming a batched reshape "should" give the same answer?
answer_1 = """
"""

# 2. Part 2 found all 12 heads sitting within about a percentage point of
#    uniform (mean max weight 0.257 vs. 0.25) on an untrained "sat" query,
#    nothing like slide 11's trained 70%/75% split. If you trained this exact
#    MultiHeadAttention module on real data, what would you expect to change
#    about the NUMBERS in that per-head table, and what would stay exactly
#    the same about the CODE?
answer_2 = """
"""

# 3. Part 3 showed EncoderBlock preserves shape across an arbitrary stack
#    depth while still moving every token's representation a real distance
#    (cosine similarity ~0.86-0.89, not ~1.0). Why is shape-preservation
#    specifically -- as opposed to any other property a layer might have --
#    the property that makes "just add more blocks" a meaningful way to make
#    a model bigger?
answer_3 = """
"""

# 4. The Challenge measured 9.54e-07 without positional encoding and ~2.25
#    with it, about 88x session 04's own crude slot-index patch. Suppose that
#    ratio had come out the other way, with the real sine/cosine encoding
#    producing a SMALLER effect than the crude patch. Would that have meant
#    positional encoding was worse? What would you need to measure, that
#    this notebook does not, to actually answer that question?
answer_4 = """
"""

# 5. Every block in this notebook is randomly initialised and has seen zero
#    training data. Name one specific claim from today's lecture (slide 11's
#    head specialisation, slide 15's "early blocks learn syntax, deep blocks
#    learn meaning," or another) that this notebook's numbers CANNOT yet
#    confirm or deny, and explain what kind of evidence session 06 would need
#    to bring to settle it.
answer_5 = """
"""

---
*The Transformer Architecture: Multi-Head Attention, Positional Encoding, and the Encoder Block · Bluevision-ai Academy*